# Bolero Notebook 2: Train and Evaluate a Subject Identifier

**Purpose:** train a ResNet-50 classifier specifically for the ten Imagenette classes, then compare it fairly against the original ImageNet-pretrained ResNet-50 baseline.

Notebook 1 remains the visual exploration notebook. This notebook is the controlled classifier-training and evaluation track. It writes only to `outputs/bolero_classifier_run2/` and does not modify Notebook 1 outputs or its source.

### Research scope and important limitation

- Imagenette class folders provide **image-level labels**, not bounding boxes or pixel-level object masks.
- This notebook fine-tunes a **10-class image classifier** and creates target-class Grad-CAM importance maps.
- It does **not** train DETR or produce ground-truth object boundaries. To train a detector or segmentation model fairly, we would need box or mask annotations.
- A classifier trained on Imagenette should reduce the class-vocabulary mismatch for these ten classes, but it will not automatically solve background bias. We will inspect errors and use Grad-CAM as a diagnostic, not as ground truth.
- The evaluation split is used for model selection and reporting. For a final paper-quality estimate, keep a separate untouched test set or obtain another held-out set.

Run cells from top to bottom. First execution downloads the pretrained ResNet-50 weights if they are not already cached.

## 1. Configuration

Set `DATASET_ROOT` to the existing `val` folder. The notebook infers the dataset root and expects the usual Imagenette structure:

```text
imagenette2-160/
    train/
        n01440764/
        ...
    val/
        n01440764/
        ...
```

If the `train` folder is missing, the notebook stops with a clear message rather than quietly training on validation images.

In [ ]:
from pathlib import Path
import os, json, random, time, copy, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import ResNet50_Weights
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support
)

# ---- Paths ----
DATASET_ROOT = Path(r"C:\Users\srohi\VSCodeProjects\bolero_contextual_robustness\data\imagenette2-160\val")
# Notebook 2 has its own output tag. Do not point this at Notebook 1's outputs.
OUTPUT_ROOT = Path("outputs") / "bolero_classifier_run2"

# ---- Reproducibility ----
SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0              # reliable default on Windows / VS Code
HEAD_EPOCHS = 2              # train classifier head while backbone is frozen
FINETUNE_EPOCHS = 5          # fine-tune layer4 + classifier
LEARNING_RATE_HEAD = 1e-3
LEARNING_RATE_FINETUNE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 3
USE_AMP = False              # keep off by default for broad compatibility
MAX_TRAIN_IMAGES = None      # e.g. 500 to smoke-test; None means use all
MAX_VAL_IMAGES = None        # e.g. 200 to smoke-test; None means use all
SAVE_GRADCAM_EXAMPLES = 20

# ImageNet-1K class indices for the ten Imagenette synsets.
# Runtime checks below verify these names against the installed torchvision weights metadata.
IMAGENETTE_TO_IMAGENET_INDEX = {
    "n01440764": 0,    # tench
    "n02102040": 217,  # English springer
    "n02979186": 482,  # cassette player
    "n03000684": 491,  # chain saw
    "n03028079": 497,  # church
    "n03394916": 566,  # French horn
    "n03417042": 569,  # garbage truck
    "n03425413": 571,  # gas pump
    "n03445777": 574,  # golf ball
    "n03888257": 701,  # parachute
}

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATASET_ROOT = DATASET_ROOT.expanduser().resolve()
DATASET_DIR = DATASET_ROOT.parent if DATASET_ROOT.name.lower() == "val" else DATASET_ROOT
TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"
OUTPUT_ROOT = OUTPUT_ROOT.resolve()

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Dataset directory:", DATASET_DIR)
print("Training directory exists:", TRAIN_DIR.exists())
print("Validation directory exists:", VAL_DIR.exists())
print("Output directory:", OUTPUT_ROOT)

if not TRAIN_DIR.is_dir() or not VAL_DIR.is_dir():
    raise FileNotFoundError(
        "Expected both train/ and val/ folders under the Imagenette dataset root. "
        f"Looked for: {TRAIN_DIR} and {VAL_DIR}. "
        "Do not use val/ as training data. Download/extract the complete Imagenette dataset."
    )

# Clear only this notebook's tagged run directory. Never delete the parent outputs folder.
if OUTPUT_ROOT.name != "bolero_classifier_run2" or OUTPUT_ROOT.parent.name != "outputs":
    raise RuntimeError(f"Unsafe output path; expected outputs/bolero_classifier_run2, got {OUTPUT_ROOT}")
if OUTPUT_ROOT.exists():
    import shutil
    shutil.rmtree(OUTPUT_ROOT)
for sub in ["checkpoints", "metrics", "plots", "gradcam", "misclassified"]:
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)

CONFIG = {
    "seed": SEED, "image_size": IMAGE_SIZE, "batch_size": BATCH_SIZE,
    "head_epochs": HEAD_EPOCHS, "finetune_epochs": FINETUNE_EPOCHS,
    "learning_rate_head": LEARNING_RATE_HEAD,
    "learning_rate_finetune": LEARNING_RATE_FINETUNE,
    "weight_decay": WEIGHT_DECAY, "device": str(DEVICE),
    "dataset_dir": str(DATASET_DIR), "output_dir": str(OUTPUT_ROOT),
    "torch_version": torch.__version__, "python_version": platform.python_version()
}
with open(OUTPUT_ROOT / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

## 2. Load Imagenette and check the class mapping

The train and validation folders must contain matching class directories. We use ImageNet normalization for both models. Training augmentation is modest and label-preserving; validation uses deterministic preprocessing.

In [ ]:
weights = ResNet50_Weights.IMAGENET1K_V2
imagenet_categories = weights.meta["categories"]

expected_category_fragments = {
    "n01440764": "tench",
    "n02102040": "springer",
    "n02979186": "cassette player",
    "n03000684": "chain saw",
    "n03028079": "church",
    "n03394916": "french horn",
    "n03417042": "garbage truck",
    "n03425413": "gas pump",
    "n03445777": "golf ball",
    "n03888257": "parachute",
}

for synset, idx in IMAGENETTE_TO_IMAGENET_INDEX.items():
    actual = imagenet_categories[idx].lower()
    expected = expected_category_fragments[synset]
    if expected not in actual and not (expected == "springer" and "springer" in actual):
        raise ValueError(
            f"ImageNet mapping check failed for {synset}: index {idx} is '{actual}', "
            f"expected a category containing '{expected}'. Stop rather than report a misleading baseline."
        )

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=weights.transforms().mean, std=weights.transforms().std),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=weights.transforms().mean, std=weights.transforms().std),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
train_eval_dataset = datasets.ImageFolder(TRAIN_DIR, transform=eval_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=eval_transform)

if train_dataset.classes != val_dataset.classes:
    raise ValueError(f"Train/val class mismatch:\ntrain={train_dataset.classes}\nval={val_dataset.classes}")

class_names = train_dataset.classes
class_to_idx = train_dataset.class_to_idx
if len(class_names) != 10:
    print(f"WARNING: expected 10 Imagenette classes but found {len(class_names)}: {class_names}")

# Validate the folder order and synset coverage.
missing_synsets = set(IMAGENETTE_TO_IMAGENET_INDEX) - set(class_names)
if missing_synsets:
    raise ValueError(f"Expected Imagenette synset folders are missing: {sorted(missing_synsets)}")

def deterministic_subset(dataset, limit, seed):
    if limit is None or limit >= len(dataset):
        return dataset
    rng = np.random.default_rng(seed)
    selected = np.sort(rng.choice(len(dataset), size=limit, replace=False))
    return torch.utils.data.Subset(dataset, selected.tolist())

train_dataset = deterministic_subset(train_dataset, MAX_TRAIN_IMAGES, SEED)
train_eval_dataset = deterministic_subset(train_eval_dataset, MAX_TRAIN_IMAGES, SEED)
val_dataset = deterministic_subset(val_dataset, MAX_VAL_IMAGES, SEED + 1)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
)
train_eval_loader = DataLoader(
    train_eval_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
)

class_table = pd.DataFrame({
    "folder": class_names,
    "label_index": [class_to_idx[c] for c in class_names],
    "imagenet_pretrained_index": [IMAGENETTE_TO_IMAGENET_INDEX[c] for c in class_names],
    "imagenet_category": [imagenet_categories[IMAGENETTE_TO_IMAGENET_INDEX[c]] for c in class_names],
})
display(class_table)
print(f"Training images: {len(train_dataset):,} | Validation images: {len(val_dataset):,}")
print("Training class mapping:", class_to_idx)
class_table.to_csv(OUTPUT_ROOT / "metrics" / "class_mapping.csv", index=False)

## 3. Establish the pretrained ImageNet baseline

The baseline is the unmodified ImageNet-1K ResNet-50. Since its output space has 1,000 classes while Imagenette has ten, we report two distinct measures:

1. **ImageNet top-1-to-target accuracy:** the pretrained model's highest-scoring ImageNet class is the matching Imagenette class.
2. **Restricted-ten top-1 accuracy:** compare scores only for the ten mapped ImageNet categories. This is a useful diagnostic, but it is not the same as ordinary ImageNet top-1 accuracy.

The fine-tuned model is evaluated in its ten-class output space. Do not compare restricted-ten accuracy to unrestricted ImageNet top-1 as though they were identical tasks.

In [ ]:
@torch.inference_mode()
def evaluate_imagenet_baseline(loader):
    model = models.resnet50(weights=weights).to(DEVICE).eval()
    imagenet_indices = torch.tensor(
        [IMAGENETTE_TO_IMAGENET_INDEX[c] for c in class_names],
        dtype=torch.long, device=DEVICE
    )
    target_names = []
    target_indices = []
    full_top1_correct = []
    restricted_correct = []
    full_predictions = []
    restricted_predictions = []
    confidences = []

    # Subset wrappers preserve the deterministic index order but don't expose labels in a single tensor.
    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        full_pred = logits.argmax(dim=1)
        restricted_logits = logits.index_select(1, imagenet_indices)
        restricted_pred_local = restricted_logits.argmax(dim=1)
        restricted_pred = torch.tensor(
            [IMAGENETTE_TO_IMAGENET_INDEX[class_names[int(i)]] for i in restricted_pred_local.cpu()],
            device=DEVICE
        )
        expected_imagenet = imagenet_indices[labels]
        full_top1_correct.extend((full_pred == expected_imagenet).cpu().tolist())
        restricted_correct.extend((restricted_pred == expected_imagenet).cpu().tolist())
        full_predictions.extend(full_pred.cpu().tolist())
        restricted_predictions.extend(restricted_pred_local.cpu().tolist())
        confidences.extend(probs.max(dim=1).values.cpu().tolist())

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return {
        "imagenet_unrestricted_top1_accuracy": float(np.mean(full_top1_correct)),
        "imagenet_restricted10_top1_accuracy": float(np.mean(restricted_correct)),
        "mean_unrestricted_top1_confidence": float(np.mean(confidences)),
        "n": len(full_top1_correct),
    }

baseline_metrics = evaluate_imagenet_baseline(val_loader)
print(json.dumps(baseline_metrics, indent=2))
with open(OUTPUT_ROOT / "metrics" / "pretrained_baseline.json", "w", encoding="utf-8") as f:
    json.dump(baseline_metrics, f, indent=2)

## 4. Build the ten-class ResNet-50

We retain the pretrained visual features but replace the final fully connected layer with ten outputs. Training proceeds in two phases:

1. Train the new classifier head with the backbone frozen.
2. Fine-tune `layer4` and the classifier with a lower learning rate.

This is a practical starting point, not a claim that it is the optimal training recipe. Validation loss controls checkpoint selection and early stopping.

In [ ]:
model = models.resnet50(weights=weights)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, len(class_names))
model = model.to(DEVICE)

def set_trainable_phase(model, phase):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.fc.parameters():
        p.requires_grad = True
    if phase == "finetune":
        for p in model.layer4.parameters():
            p.requires_grad = True
    elif phase != "head":
        raise ValueError("phase must be 'head' or 'finetune'")

criterion = nn.CrossEntropyLoss()

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    running_loss = 0.0
    all_targets, all_preds = [], []
    n_seen = 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
        batch_n = labels.size(0)
        running_loss += float(loss.item()) * batch_n
        n_seen += batch_n
        all_targets.extend(labels.detach().cpu().tolist())
        all_preds.extend(logits.detach().argmax(dim=1).cpu().tolist())
    return {
        "loss": running_loss / max(n_seen, 1),
        "accuracy": accuracy_score(all_targets, all_preds),
        "n": n_seen
    }

history = []
best_val_loss = float("inf")
best_state = None
best_epoch = None
epochs_without_improvement = 0
global_epoch = 0

def train_phase(phase, epochs, lr):
    global best_val_loss, best_state, best_epoch, epochs_without_improvement, global_epoch, history
    set_trainable_phase(model, phase)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr, weight_decay=WEIGHT_DECAY
    )
    for phase_epoch in range(1, epochs + 1):
        global_epoch += 1
        start = time.perf_counter()
        train_metrics = run_epoch(model, train_loader, optimizer)
        val_metrics = run_epoch(model, val_loader, optimizer=None)
        elapsed = time.perf_counter() - start
        row = {
            "epoch": global_epoch, "phase": phase, "phase_epoch": phase_epoch,
            "train_loss": train_metrics["loss"], "train_accuracy": train_metrics["accuracy"],
            "val_loss": val_metrics["loss"], "val_accuracy": val_metrics["accuracy"],
            "seconds": elapsed
        }
        history.append(row)
        print(
            f"Epoch {global_epoch:02d} [{phase}] "
            f"train loss={row['train_loss']:.4f} acc={row['train_accuracy']:.3f} | "
            f"val loss={row['val_loss']:.4f} acc={row['val_accuracy']:.3f} | "
            f"{elapsed:.1f}s"
        )
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = global_epoch
            epochs_without_improvement = 0
            torch.save({
                "state_dict": best_state,
                "class_names": class_names,
                "class_to_idx": class_to_idx,
                "best_epoch": best_epoch,
                "best_val_loss": best_val_loss,
                "config": CONFIG
            }, OUTPUT_ROOT / "checkpoints" / "resnet50_imagenette10_best.pt")
        else:
            epochs_without_improvement += 1
        if phase == "finetune" and epochs_without_improvement >= PATIENCE:
            print(f"Early stopping after {PATIENCE} validation epochs without improvement.")
            break

train_phase("head", HEAD_EPOCHS, LEARNING_RATE_HEAD)
train_phase("finetune", FINETUNE_EPOCHS, LEARNING_RATE_FINETUNE)

if best_state is None:
    raise RuntimeError("Training did not produce a checkpoint.")
model.load_state_dict(best_state)
model.eval()

history_df = pd.DataFrame(history)
display(history_df)
history_df.to_csv(OUTPUT_ROOT / "metrics" / "training_history.csv", index=False)
print(f"Best checkpoint epoch: {best_epoch}; best validation loss: {best_val_loss:.4f}")

## 5. Plot training behaviour

Look for training accuracy rising while validation accuracy stalls or validation loss increases. That is a warning sign of overfitting. We preserve the best validation-loss checkpoint rather than blindly using the final epoch.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation")
axes[0].set_title("Loss by epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(history_df["epoch"], history_df["train_accuracy"], marker="o", label="Train")
axes[1].plot(history_df["epoch"], history_df["val_accuracy"], marker="o", label="Validation")
axes[1].set_title("Accuracy by epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "plots" / "training_curves.png", dpi=160, bbox_inches="tight")
plt.show()

## 6. Comparative validation evaluation

We evaluate the fine-tuned ten-class model on the same validation images used for the pretrained baseline. Report overall and per-class results, plus a confusion matrix. The validation set also informed checkpoint selection, so these are development results rather than a fully untouched final test estimate.

In [ ]:
@torch.inference_mode()
def predict_classifier(model, loader, dataset_for_paths=None):
    model.eval()
    ys, preds, probs_all = [], [], []
    for images, labels in loader:
        logits = model(images.to(DEVICE))
        probs = torch.softmax(logits, dim=1)
        ys.extend(labels.tolist())
        preds.extend(logits.argmax(dim=1).cpu().tolist())
        probs_all.extend(probs.cpu().numpy())
    return np.asarray(ys), np.asarray(preds), np.asarray(probs_all)

y_true, y_pred, y_prob = predict_classifier(model, val_loader)
overall_accuracy = float(accuracy_score(y_true, y_pred))
per_class_precision, per_class_recall, per_class_f1, per_class_support = precision_recall_fscore_support(
    y_true, y_pred, labels=list(range(len(class_names))), zero_division=0
)
per_class_df = pd.DataFrame({
    "class_folder": class_names,
    "class_name": [expected_category_fragments.get(c, c) for c in class_names],
    "precision": per_class_precision,
    "recall": per_class_recall,
    "f1": per_class_f1,
    "support": per_class_support,
})
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
report = classification_report(
    y_true, y_pred, labels=list(range(len(class_names))),
    target_names=class_names, output_dict=True, zero_division=0
)

print(f"Fine-tuned ResNet-50 ten-class top-1 accuracy: {overall_accuracy:.3%}")
display(per_class_df)
per_class_df.to_csv(OUTPUT_ROOT / "metrics" / "per_class_metrics.csv", index=False)
pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(
    OUTPUT_ROOT / "metrics" / "confusion_matrix.csv"
)
with open(OUTPUT_ROOT / "metrics" / "classification_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, interpolation="nearest")
ax.set_title("Fine-tuned ResNet-50 confusion matrix")
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_xticks(range(len(class_names)), class_names, rotation=75, ha="right")
ax.set_yticks(range(len(class_names)), class_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "plots" / "confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()

comparison = pd.DataFrame([
    {"model": "ImageNet pretrained ResNet-50, unrestricted top-1", "accuracy": baseline_metrics["imagenet_unrestricted_top1_accuracy"], "task_note": "1000-class prediction must match target ImageNet index"},
    {"model": "ImageNet pretrained ResNet-50, restricted to Imagenette 10", "accuracy": baseline_metrics["imagenet_restricted10_top1_accuracy"], "task_note": "scores compared only among the ten mapped ImageNet categories"},
    {"model": "Fine-tuned ResNet-50, Imagenette 10", "accuracy": overall_accuracy, "task_note": "trained ten-class output head; validation used for checkpoint selection"},
])
display(comparison)
comparison.to_csv(OUTPUT_ROOT / "metrics" / "model_comparison.csv", index=False)

## 7. Save misclassified examples

Misclassifications are especially useful for diagnosing background bias. We save a CSV with filenames, true labels, predictions, confidence, and the full-frame top prediction. If a class is often confused with another, inspect those images before changing the architecture.

In [ ]:
# Rebuild a deterministic validation dataset view to align predictions with file paths.
val_paths = [p for p, _ in val_dataset.samples] if isinstance(val_dataset, datasets.ImageFolder) else None
# When using a Subset, recover the underlying dataset paths and selected indices.
if isinstance(val_dataset, torch.utils.data.Subset):
    underlying = val_dataset.dataset
    val_paths = [underlying.samples[i][0] for i in val_dataset.indices]
elif isinstance(val_dataset, datasets.ImageFolder):
    val_paths = [p for p, _ in val_dataset.samples]

mis_rows = []
for i, (true_idx, pred_idx, prob) in enumerate(zip(y_true, y_pred, y_prob)):
    if true_idx != pred_idx:
        mis_rows.append({
            "path": val_paths[i] if val_paths is not None and i < len(val_paths) else "",
            "true_folder": class_names[int(true_idx)],
            "predicted_folder": class_names[int(pred_idx)],
            "confidence": float(prob[int(pred_idx)]),
            "true_class_probability": float(prob[int(true_idx)]),
        })
mis_df = pd.DataFrame(mis_rows)
mis_df.to_csv(OUTPUT_ROOT / "metrics" / "misclassified_examples.csv", index=False)
print(f"Misclassified validation images: {len(mis_df)} / {len(y_true)}")
display(mis_df.head(25))

# Save a contact sheet of up to 24 mistakes.
if len(mis_df):
    n_show = min(24, len(mis_df))
    cols = 4
    rows = int(np.ceil(n_show / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(14, 3.5 * rows))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, row in zip(axes, mis_rows[:n_show]):
        try:
            img = Image.open(row["path"]).convert("RGB")
            ax.imshow(img)
            ax.set_title(
                f"True: {row['true_folder']}\nPred: {row['predicted_folder']} ({row['confidence']:.2f})",
                fontsize=8
            )
            ax.axis("off")
        except Exception as exc:
            ax.set_title(f"Could not load image: {exc}")
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / "plots" / "misclassified_contact_sheet.png", dpi=150, bbox_inches="tight")
    plt.show()

## 8. Grad-CAM importance maps for the fine-tuned identifier

These maps are diagnostic overlays only. They are not fed back into the classifier, and they are not segmentation ground truth. We generate maps for the **true target class** as well as the predicted class for selected examples. Comparing them can reveal when the model's evidence appears concentrated on background context.

This implementation uses the final convolutional block (`layer4`) as the Grad-CAM target layer.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.hook_a = target_layer.register_forward_hook(self._save_activation)

    def _save_activation(self, module, inputs, output):
        self.activations = output
        # Attach to the activation tensor rather than using a module backward hook.
        # This is more robust with torchvision ResNet's in-place ReLU layers.
        output.register_hook(self._save_gradient)

    def _save_gradient(self, gradient):
        self.gradients = gradient

    def generate(self, input_tensor, class_index):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(input_tensor)
        score = logits[:, class_index].sum()
        score.backward()
        weights_cam = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((weights_cam * self.activations).sum(dim=1, keepdim=True))
        cam = torch.nn.functional.interpolate(
            cam, size=input_tensor.shape[-2:], mode="bilinear", align_corners=False
        )
        cam = cam[0, 0].detach().cpu().numpy()
        cam -= cam.min()
        cam /= (cam.max() + 1e-8)
        return cam

    def close(self):
        self.hook_a.remove()

## 9. Save final model and run summary

The checkpoint includes the class mapping and training configuration. Use the fine-tuned classifier as the ten-class identifier in later Bolero experiments, while keeping Notebook 1's original outputs unchanged.

In [ ]:
final_checkpoint_path = OUTPUT_ROOT / "checkpoints" / "resnet50_imagenette10_final.pt"
torch.save({
    "state_dict": model.state_dict(),
    "class_names": class_names,
    "class_to_idx": class_to_idx,
    "imagenette_to_imagenet_index": IMAGENETTE_TO_IMAGENET_INDEX,
    "weights_source": "ResNet50_Weights.IMAGENET1K_V2",
    "best_epoch": best_epoch,
    "best_val_loss": best_val_loss,
    "config": CONFIG,
}, final_checkpoint_path)

summary = {
    "images_train": len(train_dataset),
    "images_validation": len(val_dataset),
    "classes": class_names,
    "pretrained_imagenet_unrestricted_top1": baseline_metrics["imagenet_unrestricted_top1_accuracy"],
    "pretrained_imagenet_restricted10_top1": baseline_metrics["imagenet_restricted10_top1_accuracy"],
    "finetuned_imagenette10_top1": overall_accuracy,
    "best_epoch": best_epoch,
    "best_validation_loss": best_val_loss,
    "checkpoint": str(final_checkpoint_path),
    "output_root": str(OUTPUT_ROOT),
    "interpretation_warning": (
        "Validation was used for checkpoint selection. Treat results as development results, "
        "not a fully untouched final test estimate. Classifier heatmaps are not ground-truth masks."
    )
}
with open(OUTPUT_ROOT / "metrics" / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print("\nNotebook 2 complete. Outputs were written only to:", OUTPUT_ROOT)